# 4. Régressions : accessibilité commerciale et prix immobiliers

On étudie l'association entre l'accessibilité aux commerces alimentaires et le prix au m² des logements vendus en 2025.

Les modèles utilisent le logarithme du prix au m² et contrôlent la surface du logement, son type et son département. À partir du deuxième modèle, on ajoute également le niveau de densité urbaine (`DENS7`).

On utilisera la distance aux commerces, le nombre de commerces accessibles et l'indice d'accessibilité pondéré par la distance.

**Notations :**

- $P_i$ : prix au m² du logement $i$.
- $D_i$ : distance au commerce alimentaire le plus proche, toutes catégories confondues.
- $D_{ik}$ : distance au commerce le plus proche de la catégorie $k$.
- $N_{ik}$ : nombre de commerces accessibles de la catégorie $k$ (frontière nette).
- $A_{ik}$ : indice d'accessibilité de la catégorie $k$ (frontière dégressive).
- $S_i$ : surface réelle bâtie du logement.
- $\alpha_{T_i}$ : effet du type de logement.
- $\delta_{DEP_i}$ : effet du département.
- $\theta_{DENS_i}$ : effet du niveau de densité urbaine (`DENS7`).
- $\varepsilon_i$ : terme d'erreur.
- $\beta_0$ : constante ; $\beta_k$ et $\gamma$ : coefficients estimés.

Les catégories $k$ sont les hypermarchés, les supermarchés, les commerces de proximité et les commerces spécialisés.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from src import config

dvf = pd.read_parquet(config.DOSSIER_DONNEES / "dvf_enrichi_2025.parquet")

print("Nombre de ventes :", len(dvf))

Nombre de ventes : 810517


In [3]:
dvf_reg = dvf[dvf["prix_m2"] > 0].copy()

dvf_reg["log_prix_m2"] = np.log(dvf_reg["prix_m2"])


### M1 — Distance au commerce le plus proche

$$
\log(P_i) = \beta_0 + \beta_1 D_i + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \varepsilon_i
$$

On cherche à savoir si la distance au commerce alimentaire le plus proche est associée au prix au m², en contrôlant les caractéristiques du logement et le département.

In [ ]:
modele1 = smf.ols(
    "log_prix_m2 ~ dist_plus_proche + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.008727,6.175295e-03,1296.897898,0.000000,7.996624,8.020830
C(type_local)[T.Maison],0.006045,1.586252e-03,3.810802,0.000139,0.002936,0.009154
dist_plus_proche,-0.000068,7.372485e-07,-91.782537,0.000000,-0.000069,-0.000066
surface_reelle_bati,-0.001280,2.154345e-05,-59.408094,0.000000,-0.001322,-0.001238


R² : 0.493
Observations : 810517


In [ ]:
departements = tableau.loc[
    tableau.index.str.contains("code_departement")
]

display(departements.head(10))

Les coefficients des départements sont tous statistiquement significatifs. On les conserve comme variables de contrôle, mais on ne les affiche pas dans les résultats car ils sont nombreux et ne sont pas l'objet principal de notre étude.

In [5]:
tableau = modele1.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", round(modele1.rsquared, 3))
print("Observations :", int(modele1.nobs))

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.008727,6.175295e-03,1296.897898,0.000000,7.996624,8.020830
C(type_local)[T.Maison],0.006045,1.586252e-03,3.810802,0.000139,0.002936,0.009154
dist_plus_proche,-0.000068,7.372485e-07,-91.782537,0.000000,-0.000069,-0.000066
surface_reelle_bati,-0.001280,2.154345e-05,-59.408094,0.000000,-0.001322,-0.001238


R² : 0.493
Observations : 810517


Le coefficient de la distance au commerce le plus proche est négatif et statistiquement significatif (p < 0,001) : 100 mètres supplémentaires sont associés en moyenne à un prix au m² environ 0,68 % plus faible, toutes choses égales par ailleurs. Cette association reste modérée à courte distance, mais peut devenir importante en milieu rural, où la distance au commerce le plus proche atteint en moyenne 2,9 km en DENS7 = 7.

In [17]:
dvf_reg.loc[dvf_reg["DENS7"] == "7", "dist_plus_proche"].mean()

np.float64(2941.0107570301548)

Cependant, ce premier modèle ne contrôle pas la densité urbaine : habiter à côté d'une boulangerie dans un village n'est pas comparable à habiter dans un grand centre urbain, même loin des commerces. Le modèle M2 permettra de tenir compte de cette différence.

### M2 — Distance au commerce le plus proche et densité

$$
\log(P_i) = \beta_0 + \beta_1 D_i + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On ajoute la densité urbaine pour voir si elle modifie le lien entre la distance aux commerces et le prix au m².

In [6]:
modele2 = smf.ols(
    "log_prix_m2 ~ dist_plus_proche + surface_reelle_bati + C(type_local) + C(code_departement) + C(DENS7)",
    data=dvf_reg
).fit(cov_type="HC3")

In [7]:

tableau = modele2.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", round(modele2.rsquared, 3))
print("Observations :", int(modele2.nobs))

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.089043,5.920646e-03,1366.243273,0.000000e+00,8.077439,8.100648
C(type_local)[T.Maison],0.062855,1.661106e-03,37.839169,0.000000e+00,0.059599,0.066111
C(DENS7)[T.2],-0.111204,1.781365e-03,-62.426483,0.000000e+00,-0.114696,-0.107713
C(DENS7)[T.3],-0.096968,2.590863e-03,-37.426934,1.425398e-306,-0.102046,-0.091890
C(DENS7)[T.4],0.014579,2.106453e-03,6.921123,4.480778e-12,0.010450,0.018708
C(DENS7)[T.5],-0.209930,2.127747e-03,-98.663193,0.000000e+00,-0.214101,-0.205760
C(DENS7)[T.6],-0.284730,2.562359e-03,-111.120123,0.000000e+00,-0.289752,-0.279707
C(DENS7)[T.7],-0.464222,5.091231e-03,-91.180723,0.000000e+00,-0.474201,-0.454243
dist_plus_proche,-0.000019,8.765101e-07,-22.026590,1.601889e-107,-0.000021,-0.000018
surface_reelle_bati,-0.001337,2.136591e-05,-62.577104,0.000000e+00,-0.001379,-0.001295


R² : 0.509
Observations : 810517


La distance au commerce le plus proche reste négative et statistiquement significative (p < 0,001), mais son coefficient diminue par rapport à M1 : 100 mètres supplémentaires sont désormais associés à un prix au m² environ 0,19 % plus faible, contre 0,68 % auparavant.

Cela suggère qu'une partie de l'association observée dans M1 était liée aux différences de densité urbaine. Les coefficients de `DENS7` sont également significatifs et montrent des écarts de prix importants entre les territoires, notamment en milieu rural.

Il serait donc intéressant de tester une interaction entre la distance et la densité, afin de vérifier si l'association entre proximité commerciale et prix immobiliers varie selon le type de territoire.

### M3 — Interaction entre distance et densité urbaine

$$
\log(P_i) = \beta_0 + \beta_1 D_i + \theta_{DENS_i} + \lambda_{DENS_i}D_i + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \varepsilon_i
$$

On ajoute une interaction entre la distance au commerce le plus proche et la densité urbaine pour vérifier si leur association avec les prix immobiliers varie selon le type de territoire.

In [18]:
modele3 = smf.ols(
    "log_prix_m2 ~ dist_plus_proche * C(DENS7) + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")

In [19]:

tableau = modele3.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", round(modele3.rsquared, 3))
print("Observations :", int(modele3.nobs))

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.063168,0.005977,1349.139065,0.000000e+00,8.051455,8.074882
C(DENS7)[T.2],-0.121216,0.002651,-45.724225,0.000000e+00,-0.126412,-0.116020
C(DENS7)[T.3],-0.083073,0.003587,-23.159839,1.157070e-118,-0.090104,-0.076043
C(DENS7)[T.4],0.040101,0.003091,12.974809,1.700203e-38,0.034044,0.046159
C(DENS7)[T.5],-0.186354,0.002821,-66.063164,0.000000e+00,-0.191883,-0.180825
C(DENS7)[T.6],-0.211907,0.003279,-64.625672,0.000000e+00,-0.218334,-0.205481
C(DENS7)[T.7],-0.449121,0.007983,-56.259482,0.000000e+00,-0.464767,-0.433474
C(type_local)[T.Maison],0.049141,0.001694,29.017305,3.980393e-185,0.045822,0.052461
dist_plus_proche,0.000121,0.000005,24.255919,5.726540e-130,0.000111,0.000130
dist_plus_proche:C(DENS7)[T.2],-0.000002,0.000007,-0.241284,8.093352e-01,-0.000016,0.000013


R² : 0.51
Observations : 810517


L'interaction montre que l'association entre la distance aux commerces et les prix immobiliers varie selon la densité urbaine. Dans les grands centres urbains, être plus éloigné d'un commerce est associé à des prix plus élevés (+1,22 % pour 100 mètres), tandis que cette relation devient négative dans les zones rurales les moins denses.

Une hypothèse serait que, dans les centres urbains où les commerces restent globalement accessibles, les logements un peu plus éloignés sont valorisés car ils sont potentiellement dans des quartiers plus calmes. À l'inverse, en ruralité, où le commerce le plus proche se situe en moyenne à 2,9 km en DENS7 = 7, sa proximité apparait comme un avantage plus important.

### M4 — Distance au commerce le plus proche par catégorie

$$
\log(P_i) = \beta_0 + \sum_{k=1}^{4}\beta_k D_{ik} + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On distingue les quatre catégories de commerces pour déterminer si le prix de l'immobiliers est associé différement à la distance selon leur type.

In [ ]:
modele4 = smf.ols(
    "log_prix_m2 ~ dist_hyper + dist_super + dist_proximite + dist_specialise + surface_reelle_bati + C(type_local) + C(code_departement) + C(DENS7)",
    data=dvf_reg
).fit(cov_type="HC3")

In [ ]:
tableau = modele4.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", round(modele4.rsquared, 3))
print("Observations :", int(modele4.nobs))

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,8.110321,5.914623e-03,1371.232055,0.000000e+00,8.098728,8.121913
C(type_local)[T.Maison],0.072259,1.660513e-03,43.515901,0.000000e+00,0.069004,0.075513
C(DENS7)[T.2],-0.105549,1.787273e-03,-59.055749,0.000000e+00,-0.109052,-0.102046
C(DENS7)[T.3],-0.068992,2.667753e-03,-25.861492,1.806857e-147,-0.074221,-0.063763
C(DENS7)[T.4],0.042430,2.156102e-03,19.679011,3.263072e-86,0.038204,0.046656
C(DENS7)[T.5],-0.133225,2.510016e-03,-53.077467,0.000000e+00,-0.138145,-0.128306
C(DENS7)[T.6],-0.162770,3.223310e-03,-50.497711,0.000000e+00,-0.169087,-0.156452
C(DENS7)[T.7],-0.285326,5.880638e-03,-48.519599,0.000000e+00,-0.296852,-0.273800
dist_hyper,-0.000004,1.700054e-07,-25.771121,1.869275e-146,-0.000005,-0.000004
dist_super,-0.000013,3.971260e-07,-33.054820,1.326307e-239,-0.000014,-0.000012


R² : 0.512
Observations : 810517


### M5 — Nombre de commerces accessibles (frontière nette)

$$
\log(P_i) = \beta_0 + \sum_{k=1}^{4}\beta_k N_{ik} + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On étudie si le nombre de commerces accessibles dans les rayons définis est associé au prix au m², sans pondération par la distance.

In [ ]:
modele5 = smf.ols(
    "log_prix_m2 ~ nb_rayon_hyper + nb_rayon_super + nb_rayon_proximite + nb_rayon_specialise + surface_reelle_bati + C(type_local) + C(code_departement) + C(DENS7)",
    data=dvf_reg
).fit(cov_type="HC3")

In [ ]:
tableau = modele5.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", round(modele5.rsquared, 3))
print("Observations :", int(modele5.nobs))

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,7.975019,0.005917,1347.865367,0.000000e+00,7.963422,7.986616
C(type_local)[T.Maison],0.085665,0.001667,51.403877,0.000000e+00,0.082399,0.088932
C(DENS7)[T.2],-0.113007,0.001836,-61.539593,0.000000e+00,-0.116606,-0.109408
C(DENS7)[T.3],-0.080345,0.002589,-31.028920,2.196320e-211,-0.085420,-0.075270
C(DENS7)[T.4],0.039734,0.002115,18.788044,9.460208e-79,0.035589,0.043879
C(DENS7)[T.5],-0.186867,0.002124,-87.966999,0.000000e+00,-0.191030,-0.182703
C(DENS7)[T.6],-0.275174,0.002289,-120.203974,0.000000e+00,-0.279661,-0.270687
C(DENS7)[T.7],-0.457273,0.004606,-99.274021,0.000000e+00,-0.466301,-0.448245
nb_rayon_hyper,0.013090,0.000189,69.284454,0.000000e+00,0.012720,0.013460
nb_rayon_super,0.004127,0.000087,47.210537,0.000000e+00,0.003956,0.004299


R² : 0.519
Observations : 810517


### M6 — Indice d'accessibilité (frontière dégressive)

$$
\log(P_i) = \beta_0 + \sum_{k=1}^{4}\beta_k A_{ik} + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On cherche à savoir si l'indice d'accessibilité qui met plus de poids sur les commerces proches est associé aux prix immobiliers, et si les résultats diffèrent de ceux de la frontière nette.

In [ ]:
modele6 = smf.ols(
    "log_prix_m2 ~ acces_hyper + acces_super + acces_proximite + acces_specialise + surface_reelle_bati + C(type_local) + C(code_departement) + C(DENS7)",
    data=dvf_reg
).fit(cov_type="HC3")

In [ ]:
tableau = modele6.summary2().tables[1]
display(tableau.loc[~tableau.index.str.contains("code_departement")])

print("R² :", round(modele6.rsquared, 3))
print("Observations :", int(modele6.nobs))

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,7.992959,0.005933,1347.299588,0.000000e+00,7.981331,8.004587
C(type_local)[T.Maison],0.087792,0.001675,52.405189,0.000000e+00,0.084509,0.091076
C(DENS7)[T.2],-0.140004,0.001850,-75.679506,0.000000e+00,-0.143630,-0.136379
C(DENS7)[T.3],-0.106313,0.002608,-40.766609,0.000000e+00,-0.111424,-0.101202
C(DENS7)[T.4],0.024233,0.002114,11.460756,2.076939e-30,0.020089,0.028378
C(DENS7)[T.5],-0.195787,0.002110,-92.806478,0.000000e+00,-0.199922,-0.191653
C(DENS7)[T.6],-0.275419,0.002288,-120.355639,0.000000e+00,-0.279904,-0.270934
C(DENS7)[T.7],-0.457180,0.004618,-99.003443,0.000000e+00,-0.466230,-0.448129
acces_hyper,0.026725,0.000419,63.768666,0.000000e+00,0.025904,0.027547
acces_super,0.009351,0.000217,43.192299,0.000000e+00,0.008927,0.009776


R² : 0.517
Observations : 810517
